# LiDAR Sweep Aggregation

Aggregates **5 LiDAR sweeps** around a chosen anchor frame:
2 sweeps before + anchor sweep + 2 sweeps after.

All sweeps are **ego-motion compensated**: each point cloud is transformed
to the **anchor ego frame** via the global (map) coordinate system,
so the origin stays fixed even though the vehicle was moving.

Outputs:
- **§4** — BEV plot (matplotlib, one colour per sweep)
- **§5** — Interactive 3D plot (plotly)
- **§6** — Reprojection onto the anchor camera image

## 0. Imports

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

import plotly.graph_objects as go

from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

## 1. Config

Change `DATASET` to switch between **`'nuscenes_mini'`** and **`'ecp'`**.

`N_BEFORE` / `N_AFTER` control how many extra sweeps are taken on each side
(default 2 each → 5 sweeps total).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Dataset selector  →  change this one variable to switch datasets
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'           # 'ecp'  or  'nuscenes_mini'

if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3
    FRAME_IDX    = 20      # keyframe index within the scene
elif DATASET == 'ecp':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10
    FRAME_IDX    = 570    # sample_data index for ECP (walk from first_sample_token)
else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}')

CAMERA   = 'CAM_FRONT'
N_BEFORE = 2    # sweeps to include before anchor
N_AFTER  = 2    # sweeps to include after anchor

print(f'Dataset : {DATASET}')
print(f'Scene   : {SCENE_IDX}  |  Frame: {FRAME_IDX}  |  Camera: {CAMERA}')
print(f'Sweeps  : {N_BEFORE} before + 1 anchor + {N_AFTER} after  =  {N_BEFORE + 1 + N_AFTER} total')

## 2. NuScenes Setup & Anchor Frame

In [ ]:
nusc   = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)
scene  = nusc.scene[SCENE_IDX]
print(f'Scene name: {scene["name"]}')

# ── Walk to the requested frame ───────────────────────────────────────────────
# For both datasets we follow the camera sample_data linked list, then look up
# the synchronised LIDAR_TOP sweep.
cam_sd_token = nusc.get('sample', scene['first_sample_token'])['data'][CAMERA]
for _ in range(FRAME_IDX):
    sd = nusc.get('sample_data', cam_sd_token)
    if not sd['next']:
        raise ValueError(f'FRAME_IDX={FRAME_IDX} exceeds scene length.')
    cam_sd_token = sd['next']

cam_sd   = nusc.get('sample_data', cam_sd_token)
img_path = nusc.get_sample_data_path(cam_sd_token)
cam_cal  = nusc.get('calibrated_sensor', cam_sd['calibrated_sensor_token'])
cam_ep   = nusc.get('ego_pose', cam_sd['ego_pose_token'])

K     = np.array(cam_cal['camera_intrinsic'], dtype=np.float64)
R_c2e = Quaternion(cam_cal['rotation']).rotation_matrix.astype(np.float64)
t_c2e = np.array(cam_cal['translation'], dtype=np.float64)
fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]

# Anchor ego → global transform (used for ego-motion compensation reference)
R_e2g_anchor = Quaternion(cam_ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g_anchor = np.array(cam_ep['translation'], dtype=np.float64)

img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W    = img_rgb.shape[:2]

print(f'Camera SD  : {cam_sd_token}')
print(f'Image      : {img_path}')
print(f'Resolution : {W}×{H}  |  fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')

plt.figure(figsize=(14, 5))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f'Anchor image — scene {SCENE_IDX}, frame {FRAME_IDX}, {CAMERA}')
plt.tight_layout()
plt.show()

## 3. Collect ±N LiDAR Sweeps

Finds the LiDAR sweep closest in time to the anchor camera frame, then walks
the `prev` / `next` linked list to gather `N_BEFORE` earlier and `N_AFTER` later sweeps.

Each sweep record stores its own **ego_pose** (the vehicle pose at capture time),
which is used for ego-motion compensation in §3.1.

In [ ]:
# ── Find anchor LiDAR sweep ────────────────────────────────────────────────────
cam_ts = cam_sd['timestamp']

if cam_sd['is_key_frame']:
    _sample    = nusc.get('sample', cam_sd['sample_token'])
    anchor_lid_token = _sample['data']['LIDAR_TOP']
else:
    anchor_lid_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - cam_ts),
    )['token']

print(f'Anchor LiDAR token: {anchor_lid_token}')

# ── Walk linked list, skipping duplicate file entries ─────────────────────────
def _walk_lidar_tokens(nusc, anchor_token, n_before, n_after):
    """
    Return ordered list of (relative_index, lidar_sd_token) with exactly
    n_before unique sweeps before and n_after unique sweeps after the anchor
    (or fewer if the scene boundary is reached first).

    Duplicate file entries — a known ECP→nuScenes conversion artefact where
    multiple tokens point to the same .bin — are silently skipped so they
    never consume a slot.
    """
    anchor_path = nusc.get_sample_data_path(anchor_token)
    seen_paths  = {anchor_path}

    # Walk backward: keep going until n_before unique files are found
    backward = []
    tok = nusc.get('sample_data', anchor_token)['prev']
    while tok and len(backward) < n_before:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            backward.append(tok)
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['prev']

    tokens = []
    for i, t in enumerate(reversed(backward)):
        tokens.append((-(len(backward) - i), t))

    tokens.append((0, anchor_token))

    # Walk forward: keep going until n_after unique files are found
    count = 0
    tok   = nusc.get('sample_data', anchor_token)['next']
    while tok and count < n_after:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            count += 1
            tokens.append((count, tok))
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['next']

    return tokens


sweep_tokens = _walk_lidar_tokens(nusc, anchor_lid_token, N_BEFORE, N_AFTER)

print(f'\nCollected {len(sweep_tokens)} sweep(s):')
print(f'{"idx":>4}  {"dt [ms]":>10}  {"filename"}')
for rel_idx, tok in sweep_tokens:
    sd    = nusc.get('sample_data', tok)
    dt    = (sd['timestamp'] - cam_ts) / 1000.0
    fname = Path(nusc.get_sample_data_path(tok)).name
    tag   = '  ← anchor' if rel_idx == 0 else ''
    print(f'{rel_idx:+4d}  {dt:+10.1f} ms  {fname}{tag}')


## 3.1 Load & Ego-Motion Compensate All Sweeps

Each LiDAR point is transformed to the **anchor ego frame** via:

```
p_lidar  →  p_ego_sweep  =  R_l2e @ p_lidar + t_l2e
p_global =  R_e2g_sweep  @ p_ego_sweep + t_e2g_sweep
p_ego_anchor = R_e2g_anchor.T @ (p_global - t_e2g_anchor)
```

where `R_e2g_anchor` / `t_e2g_anchor` are taken from the **camera** ego_pose at the
anchor timestamp (same vehicle, sub-millisecond difference).

In [ ]:
sweep_data = []   # list of dicts, one per sweep (all unique — dedup done in walk)

for rel_idx, tok in sweep_tokens:
    sd       = nusc.get('sample_data', tok)
    lid_path = nusc.get_sample_data_path(tok)
    lid_cal  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    ego_pose = nusc.get('ego_pose', sd['ego_pose_token'])

    R_l2e    = Quaternion(lid_cal['rotation']).rotation_matrix.astype(np.float64)
    t_l2e    = np.array(lid_cal['translation'], dtype=np.float64)
    R_e2g_sw = Quaternion(ego_pose['rotation']).rotation_matrix.astype(np.float64)
    t_e2g_sw = np.array(ego_pose['translation'], dtype=np.float64)

    pts_raw     = np.fromfile(lid_path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)

    # LiDAR → ego (at sweep time) → global → anchor ego
    pts_ego_sw  = (R_l2e @ pts_raw.T).T + t_l2e
    pts_global  = (R_e2g_sw @ pts_ego_sw.T).T + t_e2g_sw
    pts_ego_anc = (R_e2g_anchor.T @ (pts_global - t_e2g_anchor).T).T

    dt_ms = (sd['timestamp'] - cam_ts) / 1000.0
    sweep_data.append({
        'rel_idx':     rel_idx,
        'token':       tok,
        'dt_ms':       dt_ms,
        'pts_ego_anc': pts_ego_anc.astype(np.float32),
        'n_pts':       len(pts_ego_anc),
    })
    print(f'  [{rel_idx:+d}]  {len(pts_ego_anc):6,} pts  dt={dt_ms:+7.1f} ms   {Path(lid_path).name}')

total_pts = sum(s['n_pts'] for s in sweep_data)
print(f'\nLoaded {len(sweep_data)} sweep(s)  |  {total_pts:,} points total')


## 4. BEV Plot

Bird's-eye view of all aggregated sweeps in the anchor ego frame.
Each sweep has a distinct colour.  The ego vehicle is at the origin (gold diamond).

In [ ]:
# Colour palette — one colour per sweep, ordered past → future
# Using a perceptually uniform sequential map so temporal ordering is visible
_cmap_bev  = plt.colormaps.get_cmap('cool').resampled(len(sweep_data))
SWEEP_COLORS = [_cmap_bev(i) for i in range(len(sweep_data))]

# ── Subsample for performance ─────────────────────────────────────────────────
MAX_PTS_BEV = 80_000   # total across all sweeps
_per_sweep  = max(1, MAX_PTS_BEV // len(sweep_data))

fig, ax = plt.subplots(figsize=(14, 8), facecolor='#111111')
ax.set_facecolor('#111111')

legend_handles = []

for i, s in enumerate(sweep_data):
    pts   = s['pts_ego_anc']
    color = SWEEP_COLORS[i]
    step  = max(1, len(pts) // _per_sweep)
    pts_s = pts[::step]

    ax.scatter(pts_s[:, 0], pts_s[:, 1],
               s=0.3, color=color, alpha=0.6, linewidths=0, rasterized=True)

    label = ('anchor  (Δt=0)' if s['rel_idx'] == 0
             else f'sweep {s["rel_idx"]:+d}  (Δt={s["dt_ms"]:+.0f} ms)')
    legend_handles.append(mpatches.Patch(color=color, label=label))

# Ego vehicle
ax.scatter([0], [0], s=180, c='gold', marker='D', zorder=10)
legend_handles.append(mpatches.Patch(color='gold', label='ego'))

ax.set_aspect('equal')
ax.set_xlabel('X ego (forward, m)', color='white')
ax.set_ylabel('Y ego (left, m)',     color='white')
ax.tick_params(colors='white')
for sp in ax.spines.values():
    sp.set_color('gray')
ax.set_title(
    f'Aggregated LiDAR — BEV, anchor ego frame  |  {DATASET}  '
    f'scene {SCENE_IDX}  frame {FRAME_IDX}  |  {total_pts:,} pts  '
    f'({len(sweep_data)} sweeps)',
    color='white',
)
ax.legend(handles=legend_handles, loc='upper right',
          facecolor='#222222', labelcolor='white', fontsize=9)
ax.grid(True, alpha=0.12, color='white')
plt.tight_layout()
plt.show()


## 5. Interactive 3D Plot (Plotly)

Same data as the BEV, but with the Z dimension (height) included.
Points are coloured by sweep index (past → future) and can be toggled in the legend.

In [ ]:
# RGB strings for plotly, same palette as the BEV
def _rgba_to_rgb_str(c):
    r, g, b = int(c[0]*255), int(c[1]*255), int(c[2]*255)
    return f'rgb({r},{g},{b})'

SWEEP_COLORS_PL = [_rgba_to_rgb_str(SWEEP_COLORS[i]) for i in range(len(sweep_data))]

MAX_PTS_3D  = 60_000
_per_sw_3d  = max(1, MAX_PTS_3D // len(sweep_data))
RADIUS_3D   = 55.0   # metres — horizontal (XY) radius around ego
X_MIN_3D    = -5.0   # metres — drop points more than 5 m behind ego (X ego = forward)

traces = []

for i, s in enumerate(sweep_data):
    pts  = s['pts_ego_anc']
    r_xy = np.sqrt(pts[:, 0]**2 + pts[:, 1]**2)
    pts  = pts[(r_xy <= RADIUS_3D) & (pts[:, 0] >= X_MIN_3D)]

    color = SWEEP_COLORS_PL[i]
    step  = max(1, len(pts) // _per_sw_3d)
    pts_s = pts[::step]

    label = ('anchor  (Δt=0)' if s['rel_idx'] == 0
             else f'sweep {s["rel_idx"]:+d}  (Δt={s["dt_ms"]:+.0f} ms)')

    traces.append(go.Scatter3d(
        x=pts_s[:, 0], y=pts_s[:, 1], z=pts_s[:, 2],
        mode='markers',
        marker=dict(size=1.5, color=color, opacity=0.65),
        name=label,
        legendgroup=label,
    ))

# Ego marker
traces.append(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=7, color='gold', symbol='diamond'),
    name='ego',
))

fig3d = go.Figure(data=traces)
fig3d.update_layout(
    title=(f'Aggregated LiDAR — 3D, anchor ego frame  |  {DATASET}  '
           f'scene {SCENE_IDX}  frame {FRAME_IDX}  |  {total_pts:,} pts  '
           f'|  r ≤ {RADIUS_3D:.0f} m  |  x ≥ {X_MIN_3D:.0f} m'),
    scene=dict(
        xaxis_title='X ego (forward, m)',
        yaxis_title='Y ego (left, m)',
        zaxis_title='Z ego (up, m)',
        aspectmode='data',
        bgcolor='rgb(15,15,25)',
        xaxis=dict(gridcolor='rgba(255,255,255,0.08)', color='white',
                   range=[X_MIN_3D, RADIUS_3D]),
        yaxis=dict(gridcolor='rgba(255,255,255,0.08)', color='white',
                   range=[-RADIUS_3D, RADIUS_3D]),
        zaxis=dict(gridcolor='rgba(255,255,255,0.08)', color='white'),
    ),
    paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0.55)', font=dict(size=10)),
    width=1100, height=650, margin=dict(l=0, r=0, t=45, b=0),
)
fig3d.show()


## 6. Reprojection onto Anchor Camera Image

All aggregated ego-frame points are projected into the anchor camera frame:

```
p_cam = R_c2e.T @ (p_ego_anchor − t_c2e)
u = p_cam.x / p_cam.z * fx + cx
v = p_cam.y / p_cam.z * fy + cy
```

Points are coloured by **depth** (metres).  Toggle `COLOR_BY` to switch between
depth-based colouring and per-sweep colouring.

In [ ]:
COLOR_BY = 'depth'   # 'depth'  or  'sweep'

# ── Concatenate all sweeps, tagging each point with its sweep index (0-based) ─
all_pts    = np.concatenate([s['pts_ego_anc'] for s in sweep_data], axis=0)
all_sw_idx = np.concatenate([
    np.full(s['n_pts'], i, dtype=np.int32)
    for i, s in enumerate(sweep_data)
], axis=0)

# ── Project to camera frame ───────────────────────────────────────────────────
pts_cam = (R_c2e.T @ (all_pts.astype(np.float64) - t_c2e).T).T   # (N, 3)  R3 camera

front  = pts_cam[:, 2] > 0
u_all  = pts_cam[front, 0] / pts_cam[front, 2] * fx + cx
v_all  = pts_cam[front, 1] / pts_cam[front, 2] * fy + cy
Z_all  = pts_cam[front, 2].astype(np.float32)
sw_all = all_sw_idx[front]

in_img = (u_all >= 0) & (u_all < W) & (v_all >= 0) & (v_all < H)
u_vis  = u_all[in_img].astype(np.float32)
v_vis  = v_all[in_img].astype(np.float32)
Z_vis  = Z_all[in_img]
sw_vis = sw_all[in_img]

print(f'Visible points in {CAMERA}: {len(u_vis):,}  '
      f'(depth {Z_vis.min():.1f}–{Z_vis.max():.1f} m)')

# ── Render ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6), facecolor='#111111')
ax.set_facecolor('#111111')
ax.imshow(img_rgb)

if COLOR_BY == 'depth':
    sc = ax.scatter(
        u_vis, v_vis,
        c=Z_vis, cmap='plasma',
        norm=plt.Normalize(vmin=Z_vis.min(), vmax=min(Z_vis.max(), 80.0)),
        s=2, linewidths=0, alpha=0.85,
    )
    cbar = fig.colorbar(sc, ax=ax, fraction=0.015, pad=0.01)
    cbar.set_label('Depth (m)', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

elif COLOR_BY == 'sweep':
    legend_handles = []
    for i, s in enumerate(sweep_data):
        mask  = sw_vis == i
        color = SWEEP_COLORS[i]
        ax.scatter(u_vis[mask], v_vis[mask],
                   s=2, color=color, alpha=0.75, linewidths=0, rasterized=True)
        label = ('anchor  (Δt=0)' if s['rel_idx'] == 0
                 else f'sweep {s["rel_idx"]:+d}  (Δt={s["dt_ms"]:+.0f} ms)')
        legend_handles.append(mpatches.Patch(color=color, label=label))
    ax.legend(handles=legend_handles, loc='upper right',
              facecolor='#222222', labelcolor='white', fontsize=9)

ax.axis('off')
ax.set_title(
    f'Aggregated LiDAR reprojected onto {CAMERA}  —  {len(u_vis):,} pts visible  '
    f'|  colored by {COLOR_BY}',
    color='white',
)
fig.patch.set_facecolor('#111111')
plt.tight_layout()
plt.show()
